# Weather

## Objective

Weather can affect playing conditions. I'll use Open_Meteo to get weather coinditions during games.

# Inputs

- `1.DataCleaning-R/Data/RDS/FullTeamGames.rds`

## Output

- `1.DataCleaning-R/Data/RDS/ELOScores.rds`


## Libraries

In [1]:
library(tidyverse)
library(httr)
library(jsonlite)
library(tidygeocoder)
library(here)

Warning message:
"package 'ggplot2' was built under R version 4.4.3"
Warning message:
"package 'purrr' was built under R version 4.4.3"
-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v dplyr     1.1.4     v readr     2.1.5
v forcats   1.0.0     v stringr   1.6.0
v ggplot2   4.0.1     v tibble    3.2.1
v lubridate 1.9.4     v tidyr     1.3.1
v purrr     1.2.1     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stats::lag()
i Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Warning message:
"package 'httr' was built under R version 4.4.3"

Attaching package: 'jsonlite'


The following object is masked from 'package:purrr':

    flatten


here() starts at /Users/eialnisman/Desktop/WC2026Forecast



Lets get our cities in coordinates using tidygeocoder.

In [2]:
Games <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "FullTeamGames.rds"))

Games <- Games %>%
    geocode(city_name , method="osm")

head(Games)

Passing 37 addresses to the Nominatim single address geocoder



Let's write function that uses this coordinates as well as time and day of the game to give us temperature, wind and precipitation when the ghame started.

In [ ]:
get_openmeteo_weather <- function(lat, long, match_date, match_time) {
  if (is.na(lat) || is.na(long) || is.na(match_date) || is.na(match_time)) {
    return(tibble(
      temperature_2m = NA_real_,
      precipitation = NA_real_,
      wind_speed_10m = NA_real_
    ))
  }

  res <- httr::GET(
    "https://archive-api.open-meteo.com/v1/archive",
    query = list(
      latitude = lat,
      longitude = long,
      start_date = as.character(match_date),
      end_date = as.character(match_date),
      hourly = "temperature_2m,precipitation,wind_speed_10m",
      timezone = "auto"
    )
  )

  if (httr::http_error(res)) {
    return(tibble(
      temperature_2m = NA_real_,
      precipitation = NA_real_,
      wind_speed_10m = NA_real_
    ))
  }

  dat <- jsonlite::fromJSON(httr::content(res, "text", encoding = "UTF-8"))

  target_datetime <- as.POSIXct(
    paste(match_date, match_time),
    format = "%Y-%m-%d %H:%M"
  )

  as_tibble(dat$hourly) %>%
    mutate(
      weather_datetime = as.POSIXct(time, format = "%Y-%m-%dT%H:%M"),
      time_diff = abs(as.numeric(difftime(weather_datetime, target_datetime, units = "mins")))
    ) %>%
    slice_min(time_diff, n = 1, with_ties = FALSE) %>%
    select(temperature_2m, precipitation, wind_speed_10m)
}


Games <- Games %>%
  rowwise() %>%
  mutate(
    weather = list(
      get_openmeteo_weather(
        lat = lat,
        long = long,
        match_date = match_date,
        match_time = match_time
      )
    )
  ) %>%
  unnest_wider(weather) %>%
  ungroup()

In [ ]:
head(Games)

match_id,tournament_id,stage_name,group_stage,goals_against,goals_for,team_id,opponent_id,city_name,team_name,opponent_name,match_date,match_time,last_game_date,rest_days,lat,long,temperature_2m,precipitation,wind_speed_10m
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<chr>,<date>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
M-2010-06,WC-2010,group stage,1,1,0,T-01,T-69,Polokwane,Algeria,Slovenia,2010-06-13,13:30,NA,7,-23.90583,29.46139,21.5,0,15.0
M-2010-23,WC-2010,group stage,1,0,0,T-01,T-28,Cape Town,Algeria,England,2010-06-18,20:30,2010-06-13,5,-33.92883,18.41722,14.6,0,14.9
M-2010-38,WC-2010,group stage,1,1,0,T-01,T-83,Pretoria,Algeria,United States,2010-06-23,16:00,2010-06-18,5,-25.74593,28.18791,17.7,0,4.0
M-2010-04,WC-2010,group stage,1,0,1,T-03,T-50,Johannesburg,Argentina,Nigeria,2010-06-12,16:00,NA,7,-26.20500,28.04972,15.8,0,12.0
M-2010-18,WC-2010,group stage,1,1,4,T-03,T-71,Johannesburg,Argentina,South Korea,2010-06-17,13:30,2010-06-12,5,-26.20500,28.04972,9.7,0,13.2
M-2010-35,WC-2010,group stage,1,0,2,T-03,T-33,Polokwane,Argentina,Greece,2010-06-22,20:30,2010-06-17,5,-23.90583,29.46139,11.2,0,1.8


In [ ]:
Weather <- Games %>%
    select(tournament_id, city_name, team_id, match_id, match_time, match_date, temperature_2m, precipitation, wind_speed_10m)

Lets manually check a few games to see data is correct

In [ ]:
set.seed(2026)
Weather %>%
    select(city_name, temperature_2m, wind_speed_10m, precipitation, match_date, match_time) %>%
        slice_sample(n=10)

city_name,temperature_2m,wind_speed_10m,precipitation,match_date,match_time
<chr>,<dbl>,<dbl>,<dbl>,<date>,<chr>
Fortaleza,28.2,22.1,0.1,2014-06-29,13:00
Rustenburg,12.1,6.3,0.0,2010-06-26,20:30
Doha,23.1,17.1,0.0,2022-11-21,19:00
Yekaterinburg,12.5,22.8,0.2,2018-06-15,17:00
Johannesburg,8.4,6.6,0.0,2010-07-02,20:30
Cape Town,13.0,8.8,0.0,2010-06-29,20:30
Moscow,15.2,9.7,0.0,2018-06-14,18:00
Durban,19.1,9.5,0.0,2010-06-16,16:00
Belo Horizonte,24.6,8.1,0.0,2014-06-28,13:00


Looks good.

In [ ]:
saveRDS(Weather, here("1.DataCleaning-R", "Data", "RDS", "Weather.rds"))